In [ ]:
import pandas as pd 
from datetime import datetime

In [ ]:
df_pars = pd.read_csv('parsing.csv')
df_pars.shape

In [ ]:
После парсига у нас получился датасет, в котором 20 414 строк

In [ ]:
df_pars.duplicated().sum()

In [ ]:
в этом датасете у нас 17 952 дубликатов. Такое количество пропусков получилось из-за того, что парсили сайт новостей ТАСС несколько человек. Хотя мы и пытались делать это в разное время, но пересечения все же получились. 
Дубликаты можно спокойно удалять

In [ ]:
df_parsing = df_pars.drop_duplicates()
df_parsing

In [ ]:
После удаления дубликатов у нас получился датасет, в котором 2 502 данных.
В дальнейшем мы будем использовать этот финальный датасет, который мы назвали parsing_final(1).csv

In [ ]:
df_parsing = pd.read_csv('parsing_final(1).csv')
df_parsing

In [ ]:
df_api = pd.read_csv('rbk_posts.csv')
df_api

In [ ]:
df_api.duplicated().sum()

In [ ]:
В датасете, полученном через API, у нас в целом нет дубликатов. Поэтому перейдем к пропускам

In [ ]:
df_parsing.isnall().sum()

In [ ]:
a = df_parsing.isnull().sum()
b = a/len(df_parsing)*100
b

In [ ]:
В столбце content у нас 176 пропусков из 2502 данных. Это 7% из всех данных. Для нас это не критичное количество пропусков, поэтому можно заполнить пропуски в этом столбце значением UNKNOWN

In [ ]:
df_parsing['content'] = df_parsing['content'].fillna('UNKNOWN')

In [ ]:
Проверим остались ли пропуски еще, после того, как мы заполнили столбец content

In [ ]:
df_parsing.isnull().sum()

In [ ]:
Пропусков не осталось, теперь посмотрим на пропуски во втором датасете

In [ ]:
df_api

In [ ]:
df_parsing.isnull().sum()

In [ ]:
c1 = df_api.isnull().sum()
d1 = c1/len(df_api)*100
d1

In [ ]:
Тут у нас есть пропуски сразу в нескольких столбцах:
text, views, reaction_detail, forwards, links, grouped_id

In [ ]:
Ноо во втором датасете у нас и 60к данных, так как данные собирались с 2024 года.
Отфильтруем данные по тем же датам, что и в 1 датасете и посмотрим на количество пропусков в нем

In [ ]:
Какие даты в первом датасете?

In [ ]:
df_parsing['date'] = pd.to_datetime(df_parsing['date'])
df_parsing

In [ ]:
df_parsing['date'].min()

In [ ]:
df_parsing['date'].max()

In [ ]:
Видим, что самая ранняя новость в нашем датасете была 4 июня 2025 года, а самая поздняя 7 апреля 2026 года
Но количество новостей всего 2,5к, что странно
Посмотрим на распределение новостей по датам

In [ ]:
df_parsing['date'] = pd.to_datetime(df_parsing['date'])
df_parsing['day'] = df_parsing['date'].dt.date
count_day = df_parsing['day'].value_counts().sort_index()

In [ ]:
pivot_table = pd.DataFrame({'Дата' : count_day.index, 'Количество новостей' : daily_counts.values})
pivot_table

In [ ]:
ага, видим, что у нас новости за 4 июня 2025 года и за 4 апреля - 7 апреля
2026 года, а за период между этими датами новости у нас отсутствуют. 
Значит во втором датасете тоже выделим эти даты

In [ ]:
df_api['date'] = pd.to_datetime(df_api['date'])

In [ ]:
df_api['date_only'] = df_api['date'].dt.date
df_api

In [ ]:
df_api['date_only'] = pd.to_datetime(df_api['date_only'])
date_need = pd.to_datetime(['2025-06-04', '2026-04-04', '2026-04-05', '2026-04-06', '2026-04-07'])
df_api_filtred = df_api[df_api['date_only'].isin(date_need)]
df_api_filtred.to_csv('rbk_posts_date.csv', index = False)

In [ ]:
df_api_date = pd.read_csv('rbk_posts_date.csv')
df_api_date

In [ ]:
итак, в этом датасете у нас всего 369 новостей за те же даты, что и в датасете, полученном через парсинг. 
Можно сделать вывод, что ТАСС публикует больше ноновстей и чаще, нежели РБК. 
Наша догадка, что это связано с более грубой цензурой на РБК ну и более официальным стилем.

Ну ничего, посмотрим сколько тут пропусков

In [ ]:
df_api_date.isnull().sum()

In [ ]:
итак, тут у нас пропуски в тех же самых столбцах, но посмотрим на их количество

In [ ]:
c = df_api_date.isnull().sum()
d = c/len(df_api_date)*100
d

In [ ]:
хмм, в столбцах links и grouped_id одни пропуски... в изначальном датасете их было тоже много в этих столбцах, 99% и 87% соответственно.

в других же столбцах маленький процент пропусков, что в этом датасете, что и в первоначальном, 
это для нас не критично, эти пропуски обработаем чуть позже, а теперь посмотрим на столбцы links и grouped_id, 
что тут вообще за данные

In [ ]:
df_api_date

In [ ]:
так, ну ссылки на тг посты нам не особо нужны для нашей бизнес задачи, да и grouped_id нам тоже не надо.

Так что смело можем удалять эти столбцы, так как они нам не нужны

In [ ]:
df_api = df_api.drop(['links', 'grouped_id'], axis = 1)
df_api

In [ ]:
df_api_date = df_api_date.drop(['links', 'grouped_id'], axis = 1)
df_api_date

In [ ]:
Смотрим дальше на пропуски

In [ ]:
c = df_api_date.isnull().sum()
d = c/len(df_api_date)*100
d

In [ ]:
c1 = df_api.isnull().sum()
d1 = c1/len(df_api)*100
d1

In [ ]:
столбец text заполняем значением "UNKNOWN" по аналогии с датасетом после парсинга. 
Так как описание новости, ее содержание нам необходимо для анализа, мы не заполняем эти пропуски каким-то другим значеним и не удаляем их, несмотря на то, что их мало (9%)

добавили позже: полностью уберем пропуски позже, так как обнаружили, что в этом столбце и еще в двух других проуски в одних строках. 
Разберемся сначала с ними (это будет чуть ниже), а потом обработаем пропуски в столбце text (станет content)

но для начала переименуем название столбца на content, как в датасете по парсингу

In [ ]:
df_api = df_api.rename(columns = {'text' : 'content'})
df_api

In [ ]:
df_api_date = df_api_date.rename(columns = {'text' : 'content'})
df_api_date

In [ ]:
# df_api['content'] = df_api['content'].fillna('UNKNOWN')
# df_api_date['content'] = df_api_date['content'].fillna('UNKNOWN')

In [ ]:
В столбцах views и forwards у нас тоже есть пропуски, но их очень мало (меньше 1%), причем их одинаковое количество. 
Посмотрим а не в одинаковых ли строках у них пропуски

In [ ]:
df_api[df_api['views'].isna()]

In [ ]:
df_api[df_api['forwards'].isna()]

In [ ]:
пропуски в этих столбцах в 76 строках, на первый взгляд кажется, что в одних и тех же. 
Чтобы это проверить, проверим количество строй с пропусками в обоих этих столбцах

In [ ]:
df_api[df_api[['views', 'forwards']].isna().any(axis = 1)]

In [ ]:
да, пропуски в одних и тех же строках в этих столбцах (количество 76 строк совпадает), но кажется, что в столбце content в этих строках тоже пропуски. 
Проверим

In [ ]:
df_api[df_api[['content', 'views', 'forwards']].isna().any(axis = 1)]

In [ ]:
да, тут тоже пропуски совпадают. 
А зачем нам тогда эти строки, если нет ни новости, ни просмотров, ни реакций и тд

Правльно, не нужны! Значит смело удаляем

In [ ]:
df_api = df_api.dropna(subset = ['content', 'views', 'forwards'], how = 'all')
df_api

In [ ]:
df_api.isnull().sum()

In [ ]:
df_api[df_api['content'].isna()]

In [ ]:
так, а тут у нас есть еще пропуски в столбце contetn, но при этом значения в столбцах views и forwards заполнены. 
Смотрим на столбец media_type и понимаем, что эти новости/посты были просто в виде фото, а не текста. 
В этом случае заполняем пропуски в столбце content на UNKNOWN, но держим в уме, что возможно именно эти строки нам для анализа не понадобятся

In [ ]:
df_api['content'] = df_api['content'].fillna('UNKNOWN')

In [ ]:
df_api.isnull().sum()

In [ ]:
Пропуски остались в одном столбце

Теперь надо сделать то же самое и для датасета rbk_posts_date, чтобы использовать второй датасет для анализа

In [ ]:
df_api_date.isnull().sum()

In [ ]:
Но тут нет пропусков в трех столбцах (views, content, forwards одновременно

In [ ]:
df_api_date['content'] = df_api_date['content'].fillna('UNKNOWN')
df_api_date

In [ ]:
df_api.isnull().sum()

In [ ]:
пропуски остались в столбце с реакциями. Тут пропусков тоже мало (около 8%)

In [ ]:
df_api[df_api['reactions_detail'].isna()]

In [ ]:
df_api[df_api[['content', 'reactions_detail']].isna().any(axis = 1)]

In [ ]:
Хммм, а пропуски в столбцах content и reactions_detail на самом деле совпадают. 
А зачем нам информация о новостях, где нет ни реакции, ни описание, только какое-то фото? 
Для нашей бизнес задачи как будто бы не очень надо, тем более пропусков мало (около 8%) так что тут тоже смело удаляем эти строки

In [ ]:
df_api = df_api.dropna(subset = ['reactions_detail'])
df_api

In [ ]:
df_api.isnull().sum

In [ ]:
df_api_date.isnull().sum()

In [ ]:
df_api_date = df_api_date.dropna(subset = ['reactions_detail'])
df_api_date

In [ ]:
df_api_date.isnull().sum()

In [ ]:
УРА! Мы обработали все пропуски во всех наших датасетах. Пойдем теперь дальше

In [ ]:
Вернемся к нашим датам, у нас там были интересные значения:

In [ ]:
pivot_table

In [ ]:
тут мы видим, что у нас возникли новости за 4 июня 2025 года, не будем на них детально смотреть, посмотрим на новости а 4 дня апреля 2026 года

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df_pivot_table = pivot_table.drop(0).reset_index(drop = True)
df_pivot_table

In [ ]:
plt.figure(figsize = (14,7))
plt.plot(df_pivot_table['Дата'], df_pivot_table['Количество новостей'], marker = 'o', linewidth = 2, markersize = 8, color = 'blue')
plt.title('Распределение новостей по дням')
plt.xlabel('Дата')
plt.ylabel('Количество новостей')
plt.show()

In [ ]:
Мы наглядно видим, что 6 апреля 2026 года у нас было пиковое значение по выпуску новостей (1099 новостей)

In [ ]:
df_parsing['date'] = pd.to_datetime(df_parsing['date'])
df_parsing['hour'] = df_parsing['date'].dt.hour
df_day = df_parsing[df_parsing['date'].dt.date == pd.to_datetime('2026-04-06').date()]
hour = df_day['hour'].value_counts().sort_index()

In [ ]:
plt.bar(hour.index, hour.values)
plt.title('Распределение выпуска новостей за 2026-04-06')
plt.show

In [ ]:
Днем с 11:00 до 13:00 6 апреля 2026 года было выпущено слишком много новостей. Посмотрим на эти новости

In [ ]:
df_parsing1 = df_parsing[df_parsing['date'].dt.date == pd.to_datetime('2026-04-06').date()]
df_parsing1

In [ ]:
start = '2026-04-06 11:00:00'
end = '2026-04-06 13:00:00'
df_parsing2 = df_parsing[(df_parsing['date'] >= start) & (df_parsing['date'] <= end)]
df_parsing2

In [ ]:
Целых 220 новостей было выпущено с 11:00 до 13:00 6 апреля 2026 на сайте ТАСС

In [ ]:
start1 = '2026-04-06 11:00:00'
end1 = '2026-04-06 13:00:00'
df_api1 = df_api[(df_api['date'] >= start) & (df_api['date'] <= end)]
df_api1

In [ ]:
и всего 7 новостей в тг РБК

